In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/train.csv').drop(columns=['PassengerId','Ticket','Cabin','Pclass','SibSp','Embarked'])
df.head(5)

,Survived,Name,Sex,Age,Parch,Fare
0,0,"Braund, Mr. Owen Harris",male,22.0,0,7.2500
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,0,71.2833
2,1,"Heikkinen, Miss. Laina",female,26.0,0,7.9250
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,0,53.1000
4,0,"Allen, Mr. William Henry",male,35.0,0,8.0500


In [3]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
df['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

## 6. Domain-Specific Construction (Titanic Context):

### a. Mother Flag:
Historical/domain knowledge about the Titanic: women with children who held the title "Mrs" and had `Parch > 0` are commonly flagged as mothers in Titanic feature engineering — this kind of feature requires actual domain understanding of the dataset's context, not just a generic formula.

In [4]:
df['IsMother'] = (
    (df['Sex'] == 'female') &
    (df['Parch'] > 0) &
    (df['Age'] > 18) &
    (df['Title'] == 'Mrs')
).astype(int)
df[df['IsMother'] == 1][['Name', 'Sex', 'Parch', 'Age', 'Title', 'IsMother']].head()

,Name,Sex,Parch,Age,Title,IsMother
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,2,27.0,Mrs,1
25,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",female,5,38.0,Mrs,1
98,"Doling, Mrs. John T (Ada Julia Bone)",female,1,34.0,Mrs,1
167,"Skoog, Mrs. William (Anna Bernhardina Karlsson)",female,4,45.0,Mrs,1
247,"Hamalainen, Mrs. William (Anna)",female,2,24.0,Mrs,1


### b. Fare Band (domain-informed binning):

Instead of arbitrary equal-width bins, using known historical class-boundary logic to bucket fares into meaningful bands (Low / Medium / High / Very High). This overlaps with Discretization from earlier, but the *bin edges themselves* are constructed using domain reasoning rather than a purely statistical method.

In [5]:
def fare_band(fare):
    if fare <= 7.91:
        return 'Low'
    elif fare <= 14.45:
        return 'Medium'
    elif fare <= 31:
        return 'High'
    else:
        return 'Very High'

df['FareBand'] = df['Fare'].apply(fare_band)
df[['Fare', 'FareBand']].head(10)

,Fare,FareBand
0,7.2500,Low
1,71.2833,Very High
2,7.9250,Medium
3,53.1000,Very High
4,8.0500,Medium
5,8.4583,Medium
6,51.8625,Very High
7,21.0750,High
8,11.1333,Medium
9,30.0708,High


In [6]:
df.head(5)

,Survived,Name,Sex,Age,Parch,Fare,Title,IsMother,FareBand
0,0,"Braund, Mr. Owen Harris",male,22.0,0,7.2500,Mr,0,Low
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,0,71.2833,Mrs,0,Very High
2,1,"Heikkinen, Miss. Laina",female,26.0,0,7.9250,Miss,0,Medium
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,0,53.1000,Mrs,0,Very High
4,0,"Allen, Mr. William Henry",male,35.0,0,8.0500,Mr,0,Medium
